# Intersection Over Union:
The core loss function in YOLO
$$ IoU= \frac{A∪B}{A∩B​}

### Core bounded box representation formats:
- YOLO : ```(x_center,y_center,w,h)```
- PyTorch : ```(xmin,ymin,xmax,ymax)```

In [ ]:
class YoloBB:
    def __init__(self,x_center, y_center, w, h) -> None:
        self.x_center = x_center
        self.y_center = y_center
        self.w = w
        self.h = h

    def area(self) -> int:
        return self.w * self.h

class PytorchBB:
    def __init__(self,xmin,ymin,xmax,ymax) -> None:
        self.xmin = xmin
        self.ymin = ymin
        self.xmax = xmax
        self.ymax = ymax
    
    def area(self) -> int:
        return (self.xmax - self.xmin) * (self.ymax - self.ymin)

def yolo_to_pytorch(box: YoloBB) -> PytorchBB:
    xmin = box.x_center - (box.w / 2)
    xmax = box.x_center + (box.w / 2)
    ymin = box.y_center - (box.h / 2)
    ymax = box.y_center + (box.h / 2)
    return PytorchBB(xmin, ymin, xmax, ymax)

def pytorch_to_yolo(box: PytorchBB) -> YoloBB:
    x_center = (box.xmin + box.xmax) / 2
    y_center = (box.ymin + box.ymax) / 2
    w = box.xmax - box.xmin
    h = box.ymax - box.ymin
    return YoloBB(x_center,y_center,w,h)

In [ ]:
def IoU(box1: PytorchBB, box2: PytorchBB ):

    xmin_I = max(box1.xmin,box2.xmin)
    ymin_I = max(box1.ymin,box2.ymin)
    xmax_I = min(box1.xmax,box2.xmax)
    ymax_I = min(box1.ymax,box2.ymax)

    w_I = max(0,xmax_I - xmin_I)
    h_I = max(0,ymax_I - ymin_I)
    area_I = w_I * h_I

    area_U = box1.area() + box2.area() - area_I 

    if area_U > 0 :
        return area_I / area_U
    else:
        return 0

In [ ]:
class Detection:
    def __init__(self,confidence,box: PytorchBB) -> None:
        self.confidence = confidence
        self.box = box

def NMS(detections: list[Detection], threshold = 0.5):
    detections = sorted(detections, key=lambda d: d.confidence, reverse= True)
    kept = []

    while detections:
        best = detections.pop(0)
        kept.append(best)

        detections = [d for d in detections if IoU(best.box, d.box) < threshold]
    
    return kept
    
